In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import pandas as pd
import numpy as np
from src.utils import *
from src.params import *
import json
from src.preprocess import *

import torch
from torch import nn
from torch.utils.data import Dataset, Subset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from pytorch_lightning import LightningDataModule, LightningModule

In [2]:
class ConvBlock(nn.Module):

    def __init__(self, in_channels, out_channels, stride= 1, kernel= 3, padding= 1):

        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(in_channels= in_channels,
                      out_channels= out_channels,
                      stride= stride,
                      padding= padding,
                      kernel_size= kernel,
                      bias= False),
            nn.BatchNorm2d(num_features= out_channels),
            nn.ReLU()
        )

    def forward(self,x):
        return self.block(x)

In [3]:
class BaselineModel(nn.Module):

    def __init__(self, n_classes, n_channels):

        super().__init__()
        self.n_classes = n_classes
        self.n_channels = n_channels

        self.block1 = nn.Sequential(
            ConvBlock(in_channels= self.n_channels,
                      out_channels= 32),
            ConvBlock(in_channels= 32,
                      out_channels= 64),
            nn.MaxPool2d(kernel_size= 3, stride= 2, padding= 1),
            nn.Dropout(0.3)
        )

        self.block2 = nn.Sequential(
            ConvBlock(in_channels= 64,
                      out_channels= 128),
            ConvBlock(in_channels= 128,
                      out_channels= 256),
            nn.MaxPool2d(kernel_size= 3, stride= 2, padding= 1),
            nn.Dropout(0.3)
        )

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d((1,1)),
            nn.Flatten(),
            nn.Linear(256, 64),
            nn.Dropout(0.3),
            nn.Linear(64, self.n_classes)
        )

    def forward(self,x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.head(x)

        return nn.functional.log_softmax(x, dim= 1)


In [4]:
test = BaselineModel(n_channels= 4, n_classes= 6)

In [ ]:
x = torch.randn(1,4, 100, 300)

In [6]:
logits = test(x)

In [7]:
logits

tensor([[-2.0403, -2.0899, -2.7527, -0.9335, -2.3860, -1.6225]],
       grad_fn=<LogSoftmaxBackward0>)

In [8]:
logits.exp().sum()

tensor(1., grad_fn=<SumBackward0>)